# TB3: First predictions

**Started:** August 5, 2026

**Last updated:** August 5, 2026

**Research questions:**

**Hypothesis:**

**Conclusion:**

**Potential Next Steps:**


**Pipeline notes**

Push scANVI-mapped BCG LT-HSCs (control day0 + treated day28) through the atlas-trained **scGen** and **IMPACT-CellOT** species transports, following Junyi notebooks **16** (HVG projection, keep mentor `.X`) and **17** (forward pass + qualitative UMAP).

**Inputs**
- `tb/data/mouse_bcg/bcg_control_scanvi_080526.h5ad`
- `tb/data/mouse_bcg/bcg_treated_scanvi_080526.h5ad`

These objects are already in **human ENSG** gene space (atlas-decoded scANVI expression in `.X`). Unlike notebook 16's mouse-symbol / ENSMUSG ortholog hop, we **match ENSG → atlas HVG-1000 directly** and zero-fill missing genes.

**Design (from nb16):** keep mentor `.X` (no count swap, no extra `normalize_total`/`log1p`). Models expect the same 1k HVG axis as `atlas_full_{flavor}`.

**Models (from nb17):** `atlas_full_{seurat_v3,pearson_residuals}` × `{scgen, impact_cellot}`.

**Kernel:** `speciesOT_env` (CellOT conda env on jzhou's home is not readable from this account; cellot imports here with a `torch.load` weights_only patch).

**HVG coverage after projection:** `pearson_residuals` 685/1000 (68.5%); `seurat_v3` 325/1000 (32.5%) — prefer pearson_residuals.

## Paths

Edit the strings in the next cell to point at different inputs, atlas HVG files, or trained model directories. The rest of the notebook reads from these variables.

In [ ]:
# --- edit these filepath strings ---

CONTROL_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/bcg_control_scanvi_080526.h5ad"
TREATED_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/bcg_treated_scanvi_080526.h5ad"

OUT_DIR = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/predictions_080526"

# CellOT library (sys.path); needed to import cellot
CELLOT_CODE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu"

# Atlas HVG gene-axis files used for projection and as AE training data.path
ATLAS_HVG = {
    "seurat_v3": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_seurat_v3_atlas_full_v07.h5ad",
    "pearson_residuals": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_atlas_full_v07.h5ad",
}

# Trained model directories (each contains config.yaml and cache/model.pt).
# IMPACT-CellOT uses the same flavor's scgen directory as its autoencoder.
MODEL_DIRS = {
    "seurat_v3": {
        "scgen": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/results/atlas_full_seurat_v3/scgen",
        "impact_cellot": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/results/atlas_full_seurat_v3/impact_cellot",
    },
    "pearson_residuals": {
        "scgen": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/results/atlas_full_pearson_residuals/scgen",
        "impact_cellot": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/results/atlas_full_pearson_residuals/impact_cellot",
    },
}


In [2]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, dpi_save=150, figsize=(5, 5), frameon=False)

CONTROL_H5AD = Path(CONTROL_H5AD)
TREATED_H5AD = Path(TREATED_H5AD)
OUT_DIR = Path(OUT_DIR)
CELLOT_CODE_DIR = Path(CELLOT_CODE_DIR)
ATLAS_HVG = {flavor: Path(p) for flavor, p in ATLAS_HVG.items()}
MODEL_DIRS = {
    flavor: {model: Path(p) for model, p in models.items()}
    for flavor, models in MODEL_DIRS.items()
}

FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FLAVORS = list(ATLAS_HVG)
MODELS = list(next(iter(MODEL_DIRS.values())))
CONDITIONS = {
    "control": CONTROL_H5AD,
    "treated": TREATED_H5AD,
}

sys.path.insert(0, str(CELLOT_CODE_DIR))

print("OUT_DIR:", OUT_DIR)
print("CELLOT_CODE_DIR exists:", CELLOT_CODE_DIR.exists())
for name, p in CONDITIONS.items():
    print(f"  input {name}: {p.exists()}  {p}")
for flavor, p in ATLAS_HVG.items():
    print(f"  atlas HVG {flavor}: {p.exists()}  {p}")
for flavor, models in MODEL_DIRS.items():
    for model, p in models.items():
        print(f"  model {flavor}/{model}: {p.exists()}  {p}")

OUT_DIR: /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/predictions_080526
DATA_DIR exists: True
  control: True  /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/bcg_control_scanvi_080526.h5ad
  treated: True  /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/bcg_treated_scanvi_080526.h5ad


## 1. Load scANVI-mapped BCG LT-HSCs (keep mentor `.X`)

Same rule as notebook 16: **do not** replace `.X` with `layers['counts']`. Tag `condition`/`species` as `mouse` for the species-transport models; keep `study` / `day` for provenance.

In [3]:
def load_bcg_mentor(path: Path, label: str) -> ad.AnnData:
    a = sc.read_h5ad(path)
    assert str(a.var_names[0]).startswith("ENSG"), (
        f"{label}: expected human ENSG var_names (atlas-decoded); got {a.var_names[0]!r}"
    )
    if sp_sparse.issparse(a.X):
        a.X = a.X.astype(np.float32)
    else:
        a.X = np.asarray(a.X, dtype=np.float32)
    a.obs_names_make_unique()
    a.obs["condition"] = "mouse"
    a.obs["species"] = "mouse"
    a.obs["bcg_condition"] = label
    if "cell_type" not in a.obs.columns and "shared_cell_type" in a.obs.columns:
        a.obs["cell_type"] = a.obs["shared_cell_type"].astype(str)

    X = a.X.toarray() if sp_sparse.issparse(a.X) else np.asarray(a.X)
    flat = X.ravel()
    print(
        f"{label}: shape={a.shape}  .X min={flat.min():.4f} max={flat.max():.4f} "
        f"mean={flat.mean():.4f}  study={dict(a.obs['study'].astype(str).value_counts()) if 'study' in a.obs else None}"
    )
    return a


adatas = {name: load_bcg_mentor(path, name) for name, path in CONDITIONS.items()}
print("gene axis identical?", np.array_equal(adatas["control"].var_names, adatas["treated"].var_names))

control: shape=(412, 685)  .X min=0.0000 max=5.6218 mean=0.5317  study={'bcg_control': np.int64(412)}
treated: shape=(994, 685)  .X min=0.0000 max=5.1789 mean=0.5031  study={'bcg': np.int64(994)}
gene axis identical? True


### Optional: cap a single pathological max at p99.9 (notebook 16)

Only rewrites the global argmax entry; leave commented-style control via `CAP_OUTLIER=True`.

In [4]:
CAP_OUTLIER = True


def cap_global_max_at_p999(adata: ad.AnnData, label: str) -> None:
    X_w = adata.X.toarray().astype(np.float32, copy=True) if sp_sparse.issparse(adata.X) else np.asarray(adata.X, dtype=np.float32).copy()
    flat = X_w.ravel()
    p999 = float(np.percentile(flat, 99.9))
    k = int(np.argmax(flat))
    r, c = np.unravel_index(k, X_w.shape)
    old = float(flat[k])
    if old <= p999:
        print(f"{label}: no cap needed (max={old:.4g} <= p99.9={p999:.4g})")
        return
    X_w[r, c] = p999
    adata.X = X_w
    print(f"{label}: capped ({adata.obs_names[r]!r}, {adata.var_names[c]!r}) {old:.4g} -> {p999:.4g}")


if CAP_OUTLIER:
    for name, a in adatas.items():
        cap_global_max_at_p999(a, name)

control: capped ('CACAGTAAGGAATTAC-1', 'ENSG00000154620') 5.622 -> 4.594
treated: capped ('CACATTTCACCACGTG-1', 'ENSG00000154620') 5.179 -> 4.545


## 2. Project onto atlas HVG-1000 per flavor

Direct ENSG match (no BioMart ortholog step). Missing atlas HVGs → zero columns. **No further normalization.**

Expect ~685/1000 for `pearson_residuals` (the HVG set used when these files were built) and lower for `seurat_v3` — check coverage before interpreting predictions.

In [5]:
def project_to_atlas_hvg(adata: ad.AnnData, atlas_hvg_path: Path, out_path: Path) -> dict:
    atlas = sc.read_h5ad(atlas_hvg_path)
    atlas_genes = list(atlas.var_names.astype(str))
    src_index = {g: i for i, g in enumerate(adata.var_names.astype(str))}

    cols = [src_index.get(g, -1) for g in atlas_genes]
    n_present = sum(c >= 0 for c in cols)
    n_target = len(atlas_genes)

    X_src = adata.X.toarray() if sp_sparse.issparse(adata.X) else np.asarray(adata.X, dtype=np.float32)
    X_new = np.zeros((adata.n_obs, n_target), dtype=np.float32)
    for j, c in enumerate(cols):
        if c >= 0:
            X_new[:, j] = X_src[:, c]

    keep = [c for c in ["condition", "species", "cell_type", "study", "day", "bcg_condition", "_scvi_batch"] if c in adata.obs.columns]
    out = ad.AnnData(
        X=X_new,
        obs=adata.obs[keep].copy(),
        var=pd.DataFrame(index=pd.Index(atlas_genes, name="ensg")),
    )
    out.write_h5ad(out_path)
    info = {
        "out_path": str(out_path),
        "n_present": n_present,
        "n_total": n_target,
        "coverage_pct": 100.0 * n_present / n_target,
        "X_mean": float(X_new.mean()),
        "X_max": float(X_new.max()),
    }
    print(
        f"  wrote {out_path.name}: shape={out.shape}  "
        f"coverage={n_present}/{n_target} ({info['coverage_pct']:.1f}%)  "
        f"mean={info['X_mean']:.4f} max={info['X_max']:.4f}"
    )
    return info


coverage_rows = []
aligned_paths = {}  # (condition, flavor) -> path

for cond, adata in adatas.items():
    for flavor in FLAVORS:
        print(f"\n=== {cond} / {flavor} ===")
        atlas_path = ATLAS_HVG[flavor]
        out_path = OUT_DIR / f"bcg_{cond}_aligned_{flavor}.h5ad"
        info = project_to_atlas_hvg(adata, atlas_path, out_path)
        info["condition"] = cond
        info["flavor"] = flavor
        coverage_rows.append(info)
        aligned_paths[(cond, flavor)] = out_path

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(OUT_DIR / "hvg_coverage.csv", index=False)
print("\n=== Coverage summary ===")
print(coverage_df[["condition", "flavor", "n_present", "n_total", "coverage_pct"]].to_string(index=False))
low = coverage_df[coverage_df["coverage_pct"] < 80]
if len(low):
    print("\nWARNING: coverage < 80% — missing genes are zero-filled and will degrade predictions:")
    print(low[["condition", "flavor", "coverage_pct"]].to_string(index=False))


=== control / seurat_v3 ===
  wrote bcg_control_aligned_seurat_v3.h5ad: shape=(412, 1000)  coverage=325/1000 (32.5%)  mean=0.0435 max=4.3051

=== control / pearson_residuals ===
  wrote bcg_control_aligned_pearson_residuals.h5ad: shape=(412, 1000)  coverage=685/1000 (68.5%)  mean=0.3642 max=5.4902

=== treated / seurat_v3 ===
  wrote bcg_treated_aligned_seurat_v3.h5ad: shape=(994, 1000)  coverage=325/1000 (32.5%)  mean=0.0458 max=4.9622

=== treated / pearson_residuals ===
  wrote bcg_treated_aligned_pearson_residuals.h5ad: shape=(994, 1000)  coverage=685/1000 (68.5%)  mean=0.3446 max=5.0989

=== Coverage summary ===
condition            flavor  n_present  n_total  coverage_pct
  control         seurat_v3        325     1000          32.5
  control pearson_residuals        685     1000          68.5
  treated         seurat_v3        325     1000          32.5
  treated pearson_residuals        685     1000          68.5

condition            flavor  coverage_pct
  control         seu

## 3. scGen + IMPACT-CellOT forward passes

Same logic as notebook 17, run **in-process** in `speciesOT_env`:
- **scGen:** encode → + (`code_means[human] − code_means[mouse]`) from baked `scgen_shift.pt` → decode
- **IMPACT-CellOT:** encode (shared AE) → OT map `g` → decode

Writes `bcg_{control|treated}_predicted_human_via_{scgen|impact_cellot}_{flavor}.h5ad` under `OUT_DIR`.

In [ ]:
import torch

# PyTorch >= 2.6 defaults weights_only=True; CellOT checkpoints need False.
_orig_torch_load = torch.load


def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)


torch.load = _torch_load_compat

from cellot.utils import load_config
from cellot.utils.loaders import load_model
from cellot.models import load_autoencoder_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


def predict_one(flavor: str, model_name: str, bcg_path: Path, out_path: Path) -> ad.AnnData:
    results_dir = MODEL_DIRS[flavor][model_name]
    ae_dir = MODEL_DIRS[flavor]["scgen"]
    atlas_path = ATLAS_HVG[flavor]

    config = load_config(str(results_dir / "config.yaml"))
    if "ae_emb" in config.data:
        config.data.ae_emb.path = str(ae_dir)
    config.data.path = str(atlas_path)

    bcg = ad.read_h5ad(bcg_path)
    X = bcg.X.toarray() if sp_sparse.issparse(bcg.X) else np.asarray(bcg.X, dtype=np.float32)
    inputs = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    if model_name == "scgen":
        model, _ = load_autoencoder_model(
            config,
            restore=str(results_dir / "cache" / "model.pt"),
            device=DEVICE,
            input_dim=bcg.n_vars,
        )
        model.eval()
        shift_path = results_dir / "cache" / "scgen_shift.pt"
        if shift_path.exists():
            blob = torch.load(str(shift_path), map_location="cpu")
            model.code_means = blob["code_means"]
            # move tensors to device
            model.code_means = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in model.code_means.items()}
            print(f"  latent shift from {shift_path.name}")
        else:
            from cellot.utils.loaders import load_data
            from cellot.models.ae import compute_scgen_shift

            loader = load_data(config, return_as="loader")
            labels = loader.train.dataset.adata.obs[config.data.condition]
            compute_scgen_shift(model, loader.train.dataset, labels=labels, device=DEVICE)
            print("  recomputed latent shift from atlas")
        shift = model.code_means["human"] - model.code_means["mouse"]
        with torch.no_grad():
            pred = model.decode(model.encode(inputs) + shift).detach().cpu().numpy()
    elif model_name == "impact_cellot":
        latent_dim = config.model.get("latent_dim", 50)
        model, *_ = load_model(
            config,
            restore=str(results_dir / "cache" / "model.pt"),
            device=DEVICE,
            input_dim=latent_dim,
        )
        ae_config = load_config(str(ae_dir / "config.yaml"))
        ae_model, _ = load_autoencoder_model(
            ae_config,
            restore=str(ae_dir / "cache" / "model.pt"),
            device=DEVICE,
            input_dim=bcg.n_vars,
        )
        ae_model.eval()
        f, g = model
        g.eval()
        codes = ae_model.encode(inputs)
        transported = g.transport(codes.requires_grad_(True))
        pred = ae_model.decode(transported.detach()).detach().cpu().numpy()
    else:
        raise ValueError(model_name)

    out = ad.AnnData(X=pred.astype(np.float32), obs=bcg.obs.copy(), var=bcg.var.copy())
    out.obs["prediction_model"] = model_name
    out.obs["prediction_flavor"] = flavor
    out.write_h5ad(out_path)
    print(f"  wrote {out_path.name}: shape={out.shape} mean={float(out.X.mean()):.4f} max={float(out.X.max()):.4f}")
    return out


prediction_paths = {}
for cond in CONDITIONS:
    for flavor in FLAVORS:
        bcg_path = aligned_paths[(cond, flavor)]
        for model_name in MODELS:
            out_path = OUT_DIR / f"bcg_{cond}_predicted_human_via_{model_name}_{flavor}.h5ad"
            print(f"\n=== {cond} / {flavor} / {model_name} ===")
            predict_one(flavor, model_name, bcg_path, out_path)
            prediction_paths[(cond, flavor, model_name)] = out_path

print(f"\nWrote {len(prediction_paths)} prediction files → {OUT_DIR}")

## 4. Qualitative UMAP overlays

Per flavor × BCG condition: atlas mouse / atlas human / BCG mouse / scGen pred / IMPACT pred on a fresh PCA+UMAP (notebook 17 style). No quantitative eval yet.

In [ ]:
def to_dense(X):
    return np.asarray(X.toarray()) if sp_sparse.issparse(X) else np.asarray(X)


def umap_overlay(flavor: str, cond: str, out_stem: str, max_atlas_per_species: int = 1500):
    atlas = sc.read_h5ad(ATLAS_HVG[flavor])
    bcg = sc.read_h5ad(aligned_paths[(cond, flavor)])
    pred_scgen = prediction_paths.get((cond, flavor, "scgen"))
    pred_impact = prediction_paths.get((cond, flavor, "impact_cellot"))

    rng = np.random.default_rng(0)
    blocks, labels = [], []

    for sp_lab, key in [("atlas mouse", "mouse"), ("atlas human", "human")]:
        if "condition" in atlas.obs.columns:
            sub = atlas[atlas.obs["condition"].astype(str) == key]
        else:
            sub = atlas[atlas.obs["species"].astype(str) == key]
        if sub.n_obs > max_atlas_per_species:
            idx = rng.choice(sub.n_obs, max_atlas_per_species, replace=False)
            sub = sub[idx]
        if sub.n_obs:
            blocks.append(to_dense(sub.X))
            labels += [sp_lab] * sub.n_obs

    blocks.append(to_dense(bcg.X))
    labels += [f"BCG {cond}"] * bcg.n_obs

    if pred_scgen is not None and Path(pred_scgen).exists():
        s = sc.read_h5ad(pred_scgen)
        blocks.append(to_dense(s.X))
        labels += ["scGen pred"] * s.n_obs
    if pred_impact is not None and Path(pred_impact).exists():
        i = sc.read_h5ad(pred_impact)
        blocks.append(to_dense(i.X))
        labels += ["IMPACT pred"] * i.n_obs

    a = ad.AnnData(X=np.vstack(blocks), obs=pd.DataFrame({"label": labels}))
    sc.pp.pca(a, n_comps=min(50, a.n_vars - 1, a.n_obs - 1))
    sc.pp.neighbors(a, n_neighbors=min(15, a.n_obs - 1))
    sc.tl.umap(a)

    fig, ax = plt.subplots(figsize=(7, 6))
    palette = {
        "atlas mouse": "#bbbbbb",
        "atlas human": "tab:green",
        f"BCG {cond}": "tab:blue",
        "scGen pred": "tab:orange",
        "IMPACT pred": "tab:red",
    }
    for lab, color in palette.items():
        m = (a.obs["label"] == lab).values
        if not m.sum():
            continue
        ax.scatter(
            a.obsm["X_umap"][m, 0],
            a.obsm["X_umap"][m, 1],
            s=6,
            alpha=0.45,
            c=color,
            label=f"{lab} (n={m.sum()})",
            edgecolors="none",
        )
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_title(f"BCG {cond} → human  |  {flavor}", fontsize=11)
    ax.legend(fontsize=8, markerscale=2, loc="best", frameon=True)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        out = FIG_DIR / f"{out_stem}.{ext}"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"  saved {out}")
    plt.show()
    plt.close(fig)


for flavor in FLAVORS:
    for cond in CONDITIONS:
        print(f"\n=== UMAP {cond} / {flavor} ===")
        umap_overlay(flavor, cond, f"umap_{cond}_{flavor}")

print("\nDone.")